In [1]:
import pandas as pd

In [2]:
coe = pd.read_csv('/Users/keltonn/Desktop/Projects/housing_price_regressor/data/original_dataset/auxiliary/sg-coe-prices.csv')
coe

,YEAR,CATEGORY,MONTH,ROUND,QUOTA_PREMIUM,PREVAILING_QUOTA_PREMIUM,QUOTA,BIDS_RECEIVED
0,2021,a,jan,1,"$40,609","$38,137",963,"1,280"
1,2021,a,jan,2,"$43,501","$38,137",966,"1,309"
2,2021,a,feb,1,"$42,000","$39,844",885,"1,098"
3,2021,a,feb,2,"$41,001","$39,844",903,"1,226"
4,2021,a,mar,1,"$41,996","$41,397",887,"1,175"
...,...,...,...,...,...,...,...,...
534,2026,e,may,2,"$130,000",$0,256,436
535,2026,e,jun,1,"$129,000",$0,261,497
536,2026,e,jun,2,"$129,002",$0,262,487
537,2026,e,jul,1,"$129,801",$0,277,452


## Monthly COE features per category

Each row of the raw file is one COE bidding round (two per month, a few months have one). Four categories are in the file: **A** (cars up to 1600cc),
**B** (cars above 1600cc), **C** (goods vehicles and buses) and **E** (open category). There is no D (motorcycle) category.

- `PREVAILING_QUOTA_PREMIUM` is **not used**: it changes with the age of the licence (like the yield of a bond part-way to maturity), so it is not a
  market price at the time of the bid.
- The three features are `QUOTA_PREMIUM` (the price, $), `QUOTA` (licences on offer) and `BIDS_RECEIVED`. Money and thousands separators are
  removed. The monthly value of a feature is the **mean over that month's rounds**, so months with one round are comparable with months with two.
- The result has one row per year-month and one column per category x feature (`coe_<category>_<feature>`), keyed like `stock_features.csv`, and is
  written to `data/temporal_features/coe_features_monthly.csv`.

In [3]:
MONTHS = {m: i + 1 for i, m in enumerate(['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])}
FEATURES = ['QUOTA_PREMIUM', 'QUOTA', 'BIDS_RECEIVED']

clean = coe.drop(columns='PREVAILING_QUOTA_PREMIUM').copy()
for col in FEATURES:
    clean[col] = clean[col].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)
clean['RENT_APPROVAL_YEAR'] = clean['YEAR']
clean['RENT_APPROVAL_MONTH'] = clean['MONTH'].map(MONTHS)
assert clean['RENT_APPROVAL_MONTH'].notna().all() and (clean[FEATURES] > 0).all().all()

rounds = clean.groupby(['CATEGORY', 'RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH']).size()
print('rounds per category-month:', rounds.value_counts().to_dict(), '| months with one round:')
print(rounds[rounds == 1].reset_index().rename(columns={0: 'rounds'}).to_string(index=False))

rounds per category-month: {2: 268, 1: 3} | months with one round:
CATEGORY  RENT_APPROVAL_YEAR  RENT_APPROVAL_MONTH  rounds
       a                2026                    8       1
       b                2026                    8       1
       c                2026                    8       1


In [4]:
monthly = clean.groupby(['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH', 'CATEGORY'])[FEATURES].mean().unstack('CATEGORY')
monthly.columns = [f'coe_{category}_{feature.lower()}' for feature, category in monthly.columns]
monthly = monthly[sorted(monthly.columns)].reset_index()
first, last = monthly.iloc[0], monthly.iloc[-1]
print(f'{len(monthly)} months, {int(first.RENT_APPROVAL_YEAR)}-{int(first.RENT_APPROVAL_MONTH):02d} to {int(last.RENT_APPROVAL_YEAR)}-{int(last.RENT_APPROVAL_MONTH):02d}; '
      f'{monthly.shape[1] - 2} feature columns; missing values: {int(monthly.isna().sum().sum())}')
monthly.round(1).tail(8)

68 months, 2021-01 to 2026-08; 12 feature columns; missing values: 3


,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,coe_a_bids_received,coe_a_quota,coe_a_quota_premium,coe_b_bids_received,coe_b_quota,coe_b_quota_premium,coe_c_bids_received,coe_c_quota,coe_c_quota_premium,coe_e_bids_received,coe_e_quota,coe_e_quota_premium
60,2026,1,1980.5,1293.0,105755.0,1167.5,813.5,120367.0,414.0,285.0,75352.5,477.0,288.5,121445.5
61,2026,2,1978.5,1277.0,106410.5,1159.5,813.0,107945.5,461.0,293.0,74900.0,516.5,255.0,114445.0
62,2026,3,1828.5,1266.0,110055.0,1223.0,813.5,114785.0,463.0,290.0,77000.0,468.0,245.5,116504.5
63,2026,4,2473.5,1265.0,120505.0,1300.0,811.0,121000.5,516.5,292.5,81751.0,484.5,243.0,123001.5
64,2026,5,2177.0,1270.0,124509.5,1400.5,876.0,127868.5,489.5,292.5,89851.0,457.5,255.0,128850.0
65,2026,6,1922.0,1248.5,124928.0,1251.0,886.0,125245.5,421.5,292.0,93500.5,492.0,261.5,129001.0
66,2026,7,1734.5,1244.5,127500.0,1432.5,869.0,130389.5,436.0,298.5,94444.5,451.0,270.0,129886.0
67,2026,8,1522.0,1226.0,123890.0,1316.0,926.0,129910.0,465.0,315.0,91545.0,NaN,NaN,NaN


In [5]:
monthly.to_csv('/Users/keltonn/Desktop/Projects/housing_price_regressor/data/temporal_features/coe_features_monthly.csv', index=False, float_format='%.2f')
monthly.describe().T[['count', 'mean', 'min', 'max']].round(0)

,count,mean,min,max
RENT_APPROVAL_YEAR,68.0,2023.0,2021.0,2026.0
RENT_APPROVAL_MONTH,68.0,6.0,1.0,12.0
coe_a_bids_received,68.0,1321.0,617.0,2474.0
coe_a_quota,68.0,888.0,460.0,1296.0
coe_a_quota_premium,68.0,86437.0,41500.0,127500.0
coe_b_bids_received,68.0,983.0,500.0,1554.0
coe_b_quota,68.0,675.0,409.0,1020.0
coe_b_quota_premium,68.0,104101.0,46001.0,148002.0
coe_c_bids_received,68.0,312.0,86.0,675.0
coe_c_quota,68.0,186.0,46.0,376.0
